<a href="https://colab.research.google.com/github/ahmedyomiisiaka/RP-Network-Analysis/blob/main/notebooks/04_PPI_Network_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount("/content/drive")

DIRS = [
    "/content/rp_network/data/raw",
    "/content/rp_network/data/processed",
    "/content/rp_network/data/results",
    "/content/rp_network/docs",
    "/content/rp_network/figures",
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

drive_path = Path("/content/drive/MyDrive/rp_network")
colab_path = Path("/content/rp_network")

for f in drive_path.rglob("*"):
    if f.is_file():
        dest = colab_path / f.relative_to(drive_path)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dest)

print("✓ All files restored from Google Drive.")
for f in sorted(colab_path.rglob("*.csv")):
    print(f"  {f.relative_to(colab_path)}")

Mounted at /content/drive
✓ All files restored from Google Drive.
  data/processed/apoptosis_core_targets.csv
  data/processed/apoptosis_genes_final.csv
  data/processed/rp_genes_final.csv
  data/processed/rp_genes_for_network.csv
  data/raw/apoptosis_kegg.csv
  data/raw/apoptosis_reactome.csv
  data/raw/rp_genes_retigene.csv
  data/raw/rp_genes_rivolta2025.csv


# Notebook 04 — PPI Network Construction and Interacting-Partner Extraction

**Project:** Network-Based Analysis of Signalling Pathways in Retinitis Pigmentosa
**Institution:** Vilnius University, Systems Biology
**Supervisor:** Doc. Dr. Erinija Pranckevičienė
**Author:** Ahmed Yomi Isiaka
**Stage:** 4 — PPI Network Construction
**Date:** 2026-08-13

---

## 1. Objective

To identify direct protein–protein interactions involving the curated
RP-associated protein-coding gene set and distinguish original RP input
proteins from newly identified interacting partners.

Starting from 112 curated RP protein-coding genes, this notebook:

1. Retrieves direct interacting partners from STRING
2. Classifies each protein as either an original RP seed protein
   or a newly identified interacting partner
3. Constructs an interaction network and saves interaction and
   node tables
4. Investigates additional interaction evidence from IntAct
5. Documents all methodological decisions and database parameters

---
## 2. Background and Rationale

Retinitis Pigmentosa is caused by mutations in more than 100 genes.
These genes do not act in isolation — their protein products interact
with other proteins to perform their biological functions. By mapping
these protein-protein interactions (PPIs), we can:

- Identify the broader molecular network surrounding RP disease proteins
- Discover proteins that connect RP genes to downstream pathways
  such as apoptosis
- Reveal candidate intermediate proteins that may represent
  therapeutic targets

### Why STRING?

STRING (Search Tool for the Retrieval of Interacting Genes/Proteins)
is the most comprehensive PPI database available, integrating evidence
from experimental data, curated pathway databases, co-expression,
text mining, and genomic context. It covers 67.6 million proteins
across 14,094 organisms and is the standard resource for large-scale
PPI network analysis in systems biology.

### Why IntAct?

IntAct is a curated molecular interaction database maintained by
the European Bioinformatics Institute (EMBL-EBI). Unlike STRING,
which integrates multiple evidence types, IntAct focuses on
experimentally validated physical interactions with detailed
evidence annotations. It provides a complementary, higher-specificity
source of interaction evidence.

### Seed proteins vs interacting partners

All 112 RP protein-coding genes are designated as **seed proteins**
throughout the analysis. Proteins identified through PPI queries that
are not in the original seed set are classified as **interacting
partners**. Both groups are retained in the network — seed proteins
are never discarded even if they appear as interaction partners of
other seed proteins.

---
## 3. Input Dataset

| Property | Value |
|----------|-------|
| Source file | `data/processed/rp_genes_for_network.csv` |
| Number of proteins | 112 |
| Entry type | Protein-coding genes only |
| Excluded | RP17 locus, Xq27.1 locus (not protein-coding) |
| Source | RetiGene v1.13 + Rivolta et al. (2025) Table S1 |
| Confidence | 101 confirmed in both sources, 11 RetiGene-only |

The two locus entries (RP17 locus, Xq27.1 locus) from the Stage 1
master list are excluded from all protein-level network analyses
because they do not encode single protein-coding genes and cannot
be submitted to PPI databases.

---
## 4. Methodology

### 4.1 Input RP proteins
The 112 curated RP protein-coding gene symbols are loaded from
`rp_genes_for_network.csv` and used as the seed set for all
interaction queries.

### 4.2 STRING interaction retrieval
The STRING REST API is queried for direct interaction partners of
the input proteins. Each protein is queried individually to ensure
all external partners are captured.

Parameters:
- Species: Homo sapiens (NCBI taxonomy ID: 9606)
- Interaction scope: direct partners of input proteins
- Confidence threshold: **to be confirmed with supervisor**
- Network type: functional association

### 4.3 Identification of direct interacting partners
The STRING response is parsed to extract all proteins that interact
with any of the 112 input proteins. Each returned protein is
classified as:
- **Seed protein** — present in the original 112 RP input genes
- **New partner** — not present in the original input set

### 4.4 Classification of seed proteins and new partners
Overlap analysis is performed:
- How many interacting proteins are already RP seed proteins?
- How many are newly identified partners not in the input set?

This determines whether the network is primarily self-contained
among RP genes or substantially expands into new biology.

### 4.5 Interaction network construction
A network is constructed using NetworkX where:
- Nodes = all proteins (seed + new partners)
- Edges = interactions from STRING
- Node attributes = seed status, first seen step
- Edge attributes = STRING evidence scores

### 4.6 Additional interaction evidence (IntAct)
IntAct is queried for the same input proteins to retrieve
experimentally validated physical interactions. Results are
stored separately and compared with STRING results.

---
## 5. Database Information

### STRING
| Property | Value |
|----------|-------|
| Database | STRING |
| Version | v12 |
| URL | https://string-db.org |
| Species | Homo sapiens (taxonomy ID: 9606) |
| Access date | 2026-08-13 |
| API endpoint | https://string-db.org/api/tsv/interaction_partners |
| Confidence threshold | To be confirmed |
| Evidence types | Experimental, database, co-expression, text mining, neighbourhood, fusion, co-occurrence |

### IntAct
| Property | Value |
|----------|-------|
| Database | IntAct |
| Maintained by | EMBL-EBI |
| URL | https://www.ebi.ac.uk/intact |
| Access date | 2026-08-13 |
| API endpoint | https://www.ebi.ac.uk/intact/ws/interaction/findInteractions |
| Species | Homo sapiens |
| Evidence type | Experimentally validated physical interactions |

---
## 6. Methodological Decisions

| Item | Decision | Status |
|------|----------|--------|
| Input proteins | 112 curated RP protein-coding genes | Confirmed |
| Loci excluded | RP17 locus, Xq27.1 locus | Confirmed |
| Primary PPI resource | STRING | Confirmed |
| Additional PPI resource | IntAct | Confirmed |
| Interaction scope | Direct interactions with input proteins | Confirmed |
| Seed proteins | Original 112 RP proteins | Confirmed |
| New partners | Proteins not in original RP set | Confirmed |
| Species | Homo sapiens | Confirmed |
| Confidence threshold | To be confirmed with supervisor | **Pending** |
| STRING + IntAct integration | To be determined after separate retrieval | **Pending** |
| Network expansion | No iterative expansion unless requested | Confirmed |
| Output | Interaction table + node table + network file | Confirmed |

---

## 7. Output Files

| File | Description |
|------|-------------|
| `data/raw/string_interactions_raw.tsv` | Raw STRING API response |
| `data/processed/ppi_interactions.tsv` | Cleaned interaction table |
| `data/processed/ppi_nodes.tsv` | Node table with seed/partner classification |
| `data/processed/intact_interactions_raw.tsv` | Raw IntAct response |
| `data/network/rp_network.graphml` | Network in GraphML format |
| `data/network/rp_network.sif` | Network in SIF format (Cytoscape) |
| `data/network/rp_network_edgelist.tsv` | Edge list with all attributes |

---
## 8. Interpretation
*(To be completed after results are obtained)*

---

## 9. Limitations
*(To be completed after results are obtained)*

---

## 10. Next Step

After Stage 4 is complete:

**Stage 5 — Context-Specific Biological Annotation**

For each protein in the network, investigate:
1. Tissue expression — Human Protein Atlas, GTEx
2. Subcellular localisation — COMPARTMENTS database
3. Retinal cell-type specificity — scRNA-seq data

Short notes will be written documenting how information was
retrieved from each database.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 1: Install libraries and create folders
# ─────────────────────────────────────────────────────────────────

import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "requests", "pandas", "networkx"],
    check=True
)

from pathlib import Path

DIRS = [
    "/content/rp_network/data/raw",
    "/content/rp_network/data/processed",
    "/content/rp_network/data/results",
    "/content/rp_network/data/network",
    "/content/rp_network/docs",
    "/content/rp_network/figures",
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✓ Libraries installed.")
print("✓ Folder structure ready.")

✓ Libraries installed.
✓ Folder structure ready.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 2: Imports
# ─────────────────────────────────────────────────────────────────

import requests
import pandas as pd
import networkx as nx
import time
import json
from datetime import datetime

print(f"pandas   : {pd.__version__}")
print(f"networkx : {nx.__version__}")
print(f"requests : {requests.__version__}")
print("✓ All imports successful.")

pandas   : 2.2.2
networkx : 3.6.1
requests : 2.32.4
✓ All imports successful.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 3: Load and verify input RP gene list
#
# Input : data/processed/rp_genes_for_network.csv
#         Produced in Stage 1 — 112 protein-coding RP genes
#         Loci (RP17 locus, Xq27.1 locus) already excluded
#
# This cell confirms exactly what we are working with before
# making any API calls. No assumptions — verify first.
# ─────────────────────────────────────────────────────────────────

INPUT_FILE = "/content/rp_network/data/processed/rp_genes_for_network.csv"

df_input = pd.read_csv(INPUT_FILE)

print("=" * 55)
print("INPUT DATASET VERIFICATION")
print("=" * 55)
print(f"File loaded         : {INPUT_FILE}")
print(f"Total rows          : {len(df_input)}")
print(f"Columns             : {list(df_input.columns)}")
print()

# Confirm entry types
print("Entry type breakdown:")
print(df_input["entry_type"].value_counts().to_string())
print()

# Confirm no loci present
loci_check = df_input[
    df_input["gene_symbol"].str.lower().str.contains("locus", na=False)
]
print(f"Loci entries present : {len(loci_check)}", end="")
print(" ✓ (none — correctly excluded)" if len(loci_check) == 0
      else " ✗ PROBLEM — loci should be excluded")
print()

# Confidence breakdown
print("Confidence breakdown:")
print(df_input["confidence"].value_counts().to_string())
print()

# Inheritance breakdown
print("Inheritance breakdown:")
print(df_input["inheritance"].value_counts().to_string())
print()

# Extract gene symbol list for API queries
seed_genes = df_input["gene_symbol"].dropna().str.strip().tolist()
seed_genes_set = set(gene.upper() for gene in seed_genes)

print(f"Seed genes ready for query : {len(seed_genes)}")
print(f"First 10 : {seed_genes[:10]}")
print()
print("=" * 55)
print("INPUT VERIFICATION COMPLETE")
print("=" * 55)

INPUT DATASET VERIFICATION
File loaded         : /content/rp_network/data/processed/rp_genes_for_network.csv
Total rows          : 112
Columns             : ['gene_symbol', 'entry_type', 'inheritance', 'primary_category', 'rna_seq_tissue', 'scrna_broad', 'in_rivolta_2025', 'in_retigene', 'confidence', 'source_primary', 'date_compiled']

Entry type breakdown:
entry_type
Gene    112

Loci entries present : 0 ✓ (none — correctly excluded)

Confidence breakdown:
confidence
Both sources                          101
RetiGene only (post-2025 addition)     11

Inheritance breakdown:
inheritance
AR          72
AD-AR       21
AD          14
X-linked     5

Seed genes ready for query : 112
First 10 : ['AGBL5', 'AHI1', 'AIPL1', 'ARHGEF18', 'ARL2BP', 'ARL3', 'ARL6', 'BBS1', 'BBS2', 'BBS9']

INPUT VERIFICATION COMPLETE


#The confidence threshold is still "to be confirmed" with my supervisor. I run the query with 0.400 (medium confidence) first. This let me see the full picture and decide with my supervisor what threshold to apply

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 4: STRING API — test single protein query
#
# Before querying all 112 genes, we test with ONE protein (RHO)
# to confirm the API response structure is as expected.
#
# RHO (Rhodopsin) is chosen because it is the most well-known
# RP gene and its interactors are well documented in literature.
# This lets us verify the response before running the full query.
#
# NOTE: Confidence threshold is set to 0.400 (medium confidence)
# for exploratory purposes. Final threshold to be confirmed
# with supervisor before analysis is locked.
# ─────────────────────────────────────────────────────────────────

STRING_BASE    = "https://string-db.org/api"
SPECIES        = 9606          # Homo sapiens
SCORE_CUTOFF   = 400           # medium confidence — for exploration
CALLER_ID      = "vilnius_rp_thesis"

test_gene = "RHO"

print(f"Testing STRING API with single protein: {test_gene}")
print(f"Species          : Homo sapiens ({SPECIES})")
print(f"Score cutoff     : {SCORE_CUTOFF} (medium confidence — exploratory)")
print(f"API base URL     : {STRING_BASE}")
print()

url = f"{STRING_BASE}/tsv/interaction_partners"
params = {
    "identifiers"    : test_gene,
    "species"        : SPECIES,
    "required_score" : SCORE_CUTOFF,
    "limit"          : 0,
    "caller_identity": CALLER_ID,
}

r = requests.get(url, params=params, timeout=60)
print(f"Status code : {r.status_code}")
print()

if r.status_code == 200:
    from io import StringIO
    df_test = pd.read_csv(StringIO(r.text), sep="\t")
    print(f"Columns returned by STRING:")
    for col in df_test.columns:
        print(f"  - {col}")
    print()
    print(f"Interaction partners found for {test_gene} : {len(df_test)}")
    print()
    print("First 5 partners:")
    print(df_test[["preferredName_A","preferredName_B","score"]].head().to_string())
else:
    print(f"Error: {r.text[:300]}")

Testing STRING API with single protein: RHO
Species          : Homo sapiens (9606)
Score cutoff     : 400 (medium confidence — exploratory)
API base URL     : https://string-db.org/api

Status code : 200

Columns returned by STRING:
  - stringId_A
  - stringId_B
  - preferredName_A
  - preferredName_B
  - ncbiTaxonId
  - score
  - nscore
  - fscore
  - pscore
  - ascore
  - escore
  - dscore
  - tscore

Interaction partners found for RHO : 323

First 5 partners:
  preferredName_A preferredName_B  score
0             RHO             SAG  0.999
1             RHO            GRK7  0.999
2             RHO            GRK1  0.999
3             RHO           GNAT1  0.999
4             RHO            ARR3  0.993


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 5: STRING API — full query for all 112 genes
#
# Queries STRING interaction_partners endpoint for each of the
# 112 RP seed genes individually.
#
# Why one protein at a time?
#   Querying proteins individually guarantees the response contains
#   TRUE external partners of each protein, not just edges between
#   proteins we submitted together in a batch.
#
# Checkpoint saving:
#   Results are saved after every 10 proteins so no progress is
#   lost if the session disconnects.
#
# Parameters used:
#   Species        : Homo sapiens (9606)
#   Score cutoff   : 400 (medium confidence — exploratory)
#   Limit          : 0 (return all partners above threshold)
#
# NOTE: Confidence threshold (400) is for exploration only.
#   Final threshold will be confirmed with supervisor.
#
# Output: data/raw/string_interactions_raw.tsv
# ─────────────────────────────────────────────────────────────────

from io import StringIO

CHECKPOINT_FILE = "/content/rp_network/data/raw/string_checkpoint.tsv"
RAW_OUTPUT      = "/content/rp_network/data/raw/string_interactions_raw.tsv"
API_PAUSE       = 1.0   # seconds between queries — respectful to STRING server

all_rows   = []
failed     = []

print(f"Querying STRING for {len(seed_genes)} proteins...")
print(f"Score cutoff : {SCORE_CUTOFF}")
print(f"This will take approximately {len(seed_genes) * API_PAUSE / 60:.1f} minutes.")
print()

for i, gene in enumerate(seed_genes, start=1):

    print(f"  [{i:>3}/{len(seed_genes)}] {gene}", end="")

    params = {
        "identifiers"    : gene,
        "species"        : SPECIES,
        "required_score" : SCORE_CUTOFF,
        "limit"          : 0,
        "caller_identity": CALLER_ID,
    }

    try:
        r = requests.get(
            f"{STRING_BASE}/tsv/interaction_partners",
            params  = params,
            timeout = 60
        )

        if r.status_code == 200 and r.text.strip():
            df_gene = pd.read_csv(StringIO(r.text), sep="\t")

            if not df_gene.empty:
                df_gene["query_gene"]      = gene
                df_gene["query_is_seed"]   = True
                all_rows.append(df_gene)
                print(f"  → {len(df_gene)} partners")
            else:
                print(f"  → 0 partners")
        else:
            print(f"  → ERROR (status {r.status_code})")
            failed.append(gene)

    except Exception as e:
        print(f"  → EXCEPTION: {e}")
        failed.append(gene)

    # Save checkpoint every 10 genes
    if i % 10 == 0 and all_rows:
        checkpoint_df = pd.concat(all_rows, ignore_index=True)
        checkpoint_df.to_csv(CHECKPOINT_FILE, sep="\t", index=False)
        print(f"  ── Checkpoint saved ({i}/{len(seed_genes)} genes done) ──")

    time.sleep(API_PAUSE)

# Final save
if all_rows:
    df_raw = pd.concat(all_rows, ignore_index=True)
    df_raw.to_csv(RAW_OUTPUT, sep="\t", index=False)
    print()
    print(f"✓ Raw interactions saved → {RAW_OUTPUT}")
    print(f"  Total rows (before deduplication) : {len(df_raw)}")
    print(f"  Unique proteins queried            : {df_raw['query_gene'].nunique()}")
else:
    print("No interactions retrieved.")

if failed:
    print(f"\nFailed queries ({len(failed)}): {failed}")
else:
    print("All queries completed successfully.")

Querying STRING for 112 proteins...
Score cutoff : 400
This will take approximately 1.9 minutes.

  [  1/112] AGBL5  → 24 partners
  [  2/112] AHI1  → 111 partners
  [  3/112] AIPL1  → 95 partners
  [  4/112] ARHGEF18  → 39 partners
  [  5/112] ARL2BP  → 33 partners
  [  6/112] ARL3  → 167 partners
  [  7/112] ARL6  → 24 partners
  [  8/112] BBS1  → 129 partners
  [  9/112] BBS2  → 105 partners
  [ 10/112] BBS9  → 89 partners
  ── Checkpoint saved (10/112 genes done) ──
  [ 11/112] BEST1  → 107 partners
  [ 12/112] C19orf44  → 11 partners
  [ 13/112] CC2D2A  → 90 partners
  [ 14/112] CDHR1  → 73 partners
  [ 15/112] CEP162  → 68 partners
  [ 16/112] CEP250  → 88 partners
  [ 17/112] CERKL  → 83 partners
  [ 18/112] CFAP20  → 78 partners
  [ 19/112] CFAP410  → 48 partners
  [ 20/112] CFAP418  → ERROR (status 404)
  ── Checkpoint saved (20/112 genes done) ──
  [ 21/112] CLCC1  → 34 partners
  [ 22/112] CLN3  → 83 partners
  [ 23/112] CLRN1  → 62 partners
  [ 24/112] CNGA1  → 91 partners


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 6: Parse and deduplicate STRING results
#
# The raw file contains duplicate edges because both proteins in
# a pair are in our seed set (e.g. RHO queries GRK1, and GRK1
# queries RHO — same edge appears twice).
#
# Deduplication strategy:
#   Create a canonical edge key by sorting the two protein names
#   alphabetically. Keep the row with the highest score for each
#   unique pair.
#
# Also classify each protein as:
#   seed     — present in the original 112 RP input genes
#   partner  — not present in the original RP input set
# ─────────────────────────────────────────────────────────────────

from io import StringIO

# Load raw file
df_raw = pd.read_csv(RAW_OUTPUT, sep="\t")
print(f"Raw rows loaded : {len(df_raw)}")

# Create canonical edge key (alphabetically sorted pair)
df_raw["_edge_key"] = df_raw.apply(
    lambda r: tuple(sorted([r["preferredName_A"], r["preferredName_B"]])),
    axis=1
)

# Keep highest-score row per unique pair
df_dedup = (
    df_raw
    .sort_values("score", ascending=False)
    .drop_duplicates(subset=["_edge_key"])
    .drop(columns=["_edge_key"])
    .reset_index(drop=True)
)

print(f"After deduplication : {len(df_dedup)} unique edges")
print()

# Classify proteins as seed or new partner
seed_upper = seed_genes_set  # already uppercase from Cell 3

df_dedup["protein_A_is_seed"] = (
    df_dedup["preferredName_A"].str.upper().isin(seed_upper)
)
df_dedup["protein_B_is_seed"] = (
    df_dedup["preferredName_B"].str.upper().isin(seed_upper)
)

# All unique proteins in the network
all_proteins_A = set(df_dedup["preferredName_A"].dropna())
all_proteins_B = set(df_dedup["preferredName_B"].dropna())
all_proteins   = all_proteins_A | all_proteins_B

# Classify
seed_in_network    = {p for p in all_proteins if p.upper() in seed_upper}
new_partners       = {p for p in all_proteins if p.upper() not in seed_upper}

print("=" * 55)
print("NETWORK SUMMARY")
print("=" * 55)
print(f"Total unique edges           : {len(df_dedup)}")
print(f"Total unique proteins        : {len(all_proteins)}")
print(f"  Seed proteins (RP genes)   : {len(seed_in_network)}")
print(f"  New interacting partners   : {len(new_partners)}")
print()
print(f"Seed genes with NO interactions in STRING : "
      f"{112 - len(seed_in_network)}")
print()

# Score distribution
print("Score distribution:")
bins = [(0.4,0.5,"0.400-0.499"), (0.5,0.7,"0.500-0.699"),
        (0.7,0.9,"0.700-0.899"), (0.9,1.01,"0.900-1.000")]
for lo, hi, label in bins:
    n = ((df_dedup["score"] >= lo) & (df_dedup["score"] < hi)).sum()
    print(f"  {label} : {n:>5} edges")
print()

# Save cleaned interaction table
out_path = "/content/rp_network/data/processed/ppi_interactions.tsv"
df_dedup.to_csv(out_path, sep="\t", index=False)
print(f"✓ Saved cleaned interactions → {out_path}")

Raw rows loaded : 10872
After deduplication : 9835 unique edges

NETWORK SUMMARY
Total unique edges           : 9835
Total unique proteins        : 4132
  Seed proteins (RP genes)   : 105
  New interacting partners   : 4027

Seed genes with NO interactions in STRING : 7

Score distribution:
  0.400-0.499 :  3590 edges
  0.500-0.699 :  3410 edges
  0.700-0.899 :  1635 edges
  0.900-1.000 :  1200 edges

✓ Saved cleaned interactions → /content/rp_network/data/processed/ppi_interactions.tsv


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 7: Build node table
#
# Creates a table with one row per unique protein in the network.
# Each protein is annotated with:
#   protein_name   : gene symbol
#   node_type      : "Seed" (RP input gene) or "Partner" (new)
#   degree         : number of interaction partners in this network
#   is_rp_seed     : True/False
#
# This table is used for:
#   - Stage 5 context-specific annotation
#   - Network visualisation in Stage 11
# ─────────────────────────────────────────────────────────────────

node_rows = []

for protein in sorted(all_proteins):
    is_seed = protein.upper() in seed_upper

    # Count degree — how many edges involve this protein
    degree = (
        (df_dedup["preferredName_A"] == protein) |
        (df_dedup["preferredName_B"] == protein)
    ).sum()

    node_rows.append({
        "protein_name" : protein,
        "node_type"    : "Seed" if is_seed else "Partner",
        "is_rp_seed"   : is_seed,
        "degree"       : degree,
    })

df_nodes = pd.DataFrame(node_rows).sort_values(
    ["node_type", "degree"],
    ascending=[True, False]
).reset_index(drop=True)

print("=" * 55)
print("NODE TABLE SUMMARY")
print("=" * 55)
print(f"Total nodes     : {len(df_nodes)}")
print(f"  Seed proteins : {len(df_nodes[df_nodes['node_type']=='Seed'])}")
print(f"  New partners  : {len(df_nodes[df_nodes['node_type']=='Partner'])}")
print()

# Top 20 highest-degree proteins
print("Top 20 proteins by degree (most connected):")
print(df_nodes.head(20)[["protein_name","node_type","degree"]].to_string())
print()

# Degree distribution summary
seeds_df    = df_nodes[df_nodes["node_type"] == "Seed"]
partners_df = df_nodes[df_nodes["node_type"] == "Partner"]

print("Degree statistics — Seed proteins:")
print(f"  Max    : {seeds_df['degree'].max()}")
print(f"  Min    : {seeds_df['degree'].min()}")
print(f"  Mean   : {seeds_df['degree'].mean():.1f}")
print(f"  Median : {seeds_df['degree'].median():.0f}")
print()
print("Degree statistics — New partners:")
print(f"  Max    : {partners_df['degree'].max()}")
print(f"  Min    : {partners_df['degree'].min()}")
print(f"  Mean   : {partners_df['degree'].mean():.1f}")
print(f"  Median : {partners_df['degree'].median():.0f}")

# Save
node_path = "/content/rp_network/data/processed/ppi_nodes.tsv"
df_nodes.to_csv(node_path, sep="\t", index=False)
print()
print(f"✓ Saved node table → {node_path}")

NODE TABLE SUMMARY
Total nodes     : 4132
  Seed proteins : 105
  New partners  : 4027

Top 20 proteins by degree (most connected):
   protein_name node_type  degree
0         ABCA4   Partner      48
1        GUCA1B   Partner      48
2          ROM1   Partner      48
3       C8orf37   Partner      46
4           RP9   Partner      46
5        ZNF513   Partner      45
6       RPGRIP1   Partner      43
7         FSCN2   Partner      42
8        CEP290   Partner      39
9        SEMA4A   Partner      37
10        PROM1   Partner      32
11        CNGB3   Partner      30
12          CA4   Partner      29
13         LCA5   Partner      28
14        IQCB1   Partner      26
15     RPGRIP1L   Partner      26
16        CNGA3   Partner      25
17        GNAT2   Partner      24
18        PDE6C   Partner      23
19         GRK1   Partner      22

Degree statistics — Seed proteins:
  Max    : 445
  Min    : 6
  Mean   : 103.5
  Median : 88

Degree statistics — New partners:
  Max    : 48
  Min    :

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 8: Important biological observation
#
# The top connected partner proteins are well-known retinal disease
# genes not included in our 112-gene RP seed set.
# This is documented here as a key scientific observation.
#
# Known retinal disease genes appearing as partners:
#   ABCA4   — Stargardt disease, also causes RP (RP19)
#   GUCA1B  — Cone-rod dystrophy
#   ROM1    — Digenic RP (with PRPH2)
#   CEP290  — LCA, Senior-Loken syndrome, Joubert syndrome
#   RPGRIP1 — LCA, Cone-rod dystrophy
#   CNGA3   — Achromatopsia, Cone dystrophy
#   GNAT2   — Achromatopsia
#   SEMA4A  — RP, Cone-rod dystrophy
#   FSCN2   — RP
#   PROM1   — Stargardt disease, RP
#
# This confirms that STRING is returning biologically relevant
# interactions and that the RP network is embedded within a
# broader inherited retinal disease protein network.
# ─────────────────────────────────────────────────────────────────

# Known retinal disease genes in the partner set
known_retinal_partners = [
    "ABCA4", "GUCA1B", "ROM1", "C8orf37", "RP9", "ZNF513",
    "RPGRIP1", "FSCN2", "CEP290", "SEMA4A", "PROM1", "CNGB3",
    "CA4", "LCA5", "IQCB1", "RPGRIP1L", "CNGA3", "GNAT2",
    "PDE6C", "GRK1"
]

print("=" * 55)
print("BIOLOGICAL VALIDATION — TOP PARTNER PROTEINS")
print("=" * 55)
print()
print("The 20 highest-degree partner proteins are all known")
print("retinal disease genes not in the 112-gene RP input set.")
print()
print("This confirms:")
print("  1. STRING is returning biologically relevant interactions")
print("  2. The RP network connects to the broader inherited")
print("     retinal disease protein network")
print("  3. Our 112-gene seed set is embedded in a larger")
print("     retinal disease biology context")
print()

# Check how many partners have degree >= 5 (connected to 5+ seeds)
highly_connected = partners_df[partners_df["degree"] >= 5]
print(f"Partner proteins connected to 5+ seed genes : {len(highly_connected)}")
print(f"Partner proteins connected to 1 seed only   : {len(partners_df[partners_df['degree']==1])}")
print()

# Score breakdown at different thresholds
print("Network size at different confidence thresholds:")
print("(Relevant for supervisor discussion)")
print()
for threshold, label in [(0.4,"0.400 medium"), (0.5,"0.500"), (0.7,"0.700 high"), (0.9,"0.900 highest")]:
    df_t     = df_dedup[df_dedup["score"] >= threshold]
    proteins = set(df_t["preferredName_A"]) | set(df_t["preferredName_B"])
    seeds_t  = len([p for p in proteins if p.upper() in seed_upper])
    new_t    = len(proteins) - seeds_t
    print(f"  Score >= {label}:")
    print(f"    Edges    : {len(df_t)}")
    print(f"    Proteins : {len(proteins)} ({seeds_t} seeds + {new_t} partners)")

BIOLOGICAL VALIDATION — TOP PARTNER PROTEINS

The 20 highest-degree partner proteins are all known
retinal disease genes not in the 112-gene RP input set.

This confirms:
  1. STRING is returning biologically relevant interactions
  2. The RP network connects to the broader inherited
     retinal disease protein network
  3. Our 112-gene seed set is embedded in a larger
     retinal disease biology context

Partner proteins connected to 5+ seed genes : 377
Partner proteins connected to 1 seed only   : 2564

Network size at different confidence thresholds:
(Relevant for supervisor discussion)

  Score >= 0.400 medium:
    Edges    : 9835
    Proteins : 4132 (105 seeds + 4027 partners)
  Score >= 0.500:
    Edges    : 6245
    Proteins : 2608 (105 seeds + 2503 partners)
  Score >= 0.700 high:
    Edges    : 2835
    Proteins : 1292 (102 seeds + 1190 partners)
  Score >= 0.900 highest:
    Edges    : 1200
    Proteins : 662 (79 seeds + 583 partners)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 9: Document failed queries
#
# 7 of 112 genes returned no results from STRING.
# This is documented here with biological explanation.
# ─────────────────────────────────────────────────────────────────

failed_genes = ['CFAP418', 'RNU4-2', 'RNU6-1', 'RNU6-2',
                'RNU6-8', 'RNU6-9', 'SAXO6']

print("=" * 55)
print("FAILED QUERIES — DOCUMENTATION")
print("=" * 55)
print()

explanations = {
    "CFAP418" : "Gene symbol may differ in STRING v12. Alternative symbol: C8orf37 (note C8orf37 appears as a top partner — these may be the same protein)",
    "RNU4-2"  : "Small nuclear RNA gene — NOT a protein-coding gene. STRING is a protein interaction database and does not contain RNA genes.",
    "RNU6-1"  : "Small nuclear RNA gene — NOT protein-coding. Should be excluded from protein-level analysis.",
    "RNU6-2"  : "Small nuclear RNA gene — NOT protein-coding. Should be excluded from protein-level analysis.",
    "RNU6-8"  : "Small nuclear RNA gene — NOT protein-coding. Should be excluded from protein-level analysis.",
    "RNU6-9"  : "Small nuclear RNA gene — NOT protein-coding. Should be excluded from protein-level analysis.",
    "SAXO6"   : "Very recently added to RetiGene (post-2025). May not yet be in STRING v12.",
}

for gene, explanation in explanations.items():
    print(f"Gene    : {gene}")
    print(f"Reason  : {explanation}")
    print()

print("=" * 55)
print("METHODOLOGICAL NOTE")
print("=" * 55)
print()
print("The 5 RNU genes (RNU4-2, RNU6-1, RNU6-2, RNU6-8, RNU6-9)")
print("are small nuclear RNA genes added to RetiGene in January 2026.")
print("They encode RNA molecules, not proteins, and therefore")
print("cannot be nodes in a protein-protein interaction network.")
print()
print("Recommendation: These 5 genes should be removed from")
print("rp_genes_for_network.csv and documented as excluded from")
print("protein-level analysis. This reduces the working protein")
print("set from 112 to 107 protein-coding genes.")
print()
print("This will be flagged for my supervisor's confirmation.")

FAILED QUERIES — DOCUMENTATION

Gene    : CFAP418
Reason  : Gene symbol may differ in STRING v12. Alternative symbol: C8orf37 (note C8orf37 appears as a top partner — these may be the same protein)

Gene    : RNU4-2
Reason  : Small nuclear RNA gene — NOT a protein-coding gene. STRING is a protein interaction database and does not contain RNA genes.

Gene    : RNU6-1
Reason  : Small nuclear RNA gene — NOT protein-coding. Should be excluded from protein-level analysis.

Gene    : RNU6-2
Reason  : Small nuclear RNA gene — NOT protein-coding. Should be excluded from protein-level analysis.

Gene    : RNU6-8
Reason  : Small nuclear RNA gene — NOT protein-coding. Should be excluded from protein-level analysis.

Gene    : RNU6-9
Reason  : Small nuclear RNA gene — NOT protein-coding. Should be excluded from protein-level analysis.

Gene    : SAXO6
Reason  : Very recently added to RetiGene (post-2025). May not yet be in STRING v12.

METHODOLOGICAL NOTE

The 5 RNU genes (RNU4-2, RNU6-1, RNU6-2, 

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Cell 10: Save all outputs and update Drive
# ─────────────────────────────────────────────────────────────────

from google.colab import drive
import shutil
drive.mount("/content/drive")

# Save all output files to Drive
shutil.copytree(
    "/content/rp_network",
    "/content/drive/MyDrive/rp_network",
    dirs_exist_ok=True
)

print("=" * 55)
print("STAGE 4 — INTERIM RESULTS SUMMARY")
print("=" * 55)
print()
print("Input:")
print(f"  RP seed genes queried       : 112")
print(f"  Successfully queried        : 105")
print(f"  Failed (not in STRING)      : 7")
print()
print("Network (score >= 0.400, exploratory):")
print(f"  Total unique edges          : {len(df_dedup)}")
print(f"  Total unique proteins       : {len(all_proteins)}")
print(f"  Seed proteins in network    : {len(seed_in_network)}")
print(f"  New interacting partners    : {len(new_partners)}")
print()
print("Score distribution:")
for lo, hi, label in [(0.4,0.5,"0.400-0.499"),
                       (0.5,0.7,"0.500-0.699"),
                       (0.7,0.9,"0.700-0.899"),
                       (0.9,1.01,"0.900-1.000")]:
    n = ((df_dedup["score"] >= lo) & (df_dedup["score"] < hi)).sum()
    print(f"  {label} : {n} edges")
print()
print("Key finding:")
print("  Top partner proteins are known retinal disease genes")
print("  (ABCA4, CEP290, RPGRIP1, CNGA3, GNAT2 etc.)")
print("  confirming biological relevance of the network.")
print()
print("Pending supervisor confirmation:")
print("  1. Final confidence threshold to use")
print("  2. Handling of 5 RNU RNA genes (recommend exclusion)")
print("  3. STRING + IntAct integration strategy")
print()
print("Files saved to Drive:")
print("  data/raw/string_interactions_raw.tsv")
print("  data/processed/ppi_interactions.tsv")
print("  data/processed/ppi_nodes.tsv")
print()
print("✓ All files saved to Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 4 — INTERIM RESULTS SUMMARY

Input:
  RP seed genes queried       : 112
  Successfully queried        : 105
  Failed (not in STRING)      : 7

Network (score >= 0.400, exploratory):
  Total unique edges          : 9835
  Total unique proteins       : 4132
  Seed proteins in network    : 105
  New interacting partners    : 4027

Score distribution:
  0.400-0.499 : 3590 edges
  0.500-0.699 : 3410 edges
  0.700-0.899 : 1635 edges
  0.900-1.000 : 1200 edges

Key finding:
  Top partner proteins are known retinal disease genes
  (ABCA4, CEP290, RPGRIP1, CNGA3, GNAT2 etc.)
  confirming biological relevance of the network.

Pending supervisor confirmation:
  1. Final confidence threshold to use
  2. Handling of 5 RNU RNA genes (recommend exclusion)
  3. STRING + IntAct integration strategy

Files saved to Drive:
  data/raw/string_interactions_raw.tsv
  data/pro

In [ ]:
from google.colab import files
files.download("/content/rp_network/data/processed/ppi_interactions.tsv")
files.download("/content/rp_network/data/processed/ppi_nodes.tsv")
files.download("/content/rp_network/docs/progress_log.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Drive if not already mounted
drive.mount("/content/drive")

# Copy everything from Colab to Drive
shutil.copytree(
    "/content/rp_network",
    "/content/drive/MyDrive/rp_network",
    dirs_exist_ok=True
)

print("✓ All files saved to Google Drive.")
print()
print("Files saved:")
drive_path = Path("/content/drive/MyDrive/rp_network")
for f in sorted(drive_path.rglob("*.csv")):
    size = f.stat().st_size / 1024
    print(f"  {f.relative_to(drive_path)} ({size:.1f} KB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ All files saved to Google Drive.

Files saved:
  data/processed/apoptosis_core_targets.csv (4.1 KB)
  data/processed/apoptosis_genes_final.csv (29.7 KB)
  data/processed/rp_genes_final.csv (12.2 KB)
  data/processed/rp_genes_for_network.csv (11.8 KB)
  data/raw/apoptosis_kegg.csv (20.1 KB)
  data/raw/apoptosis_reactome.csv (16.0 KB)
  data/raw/rp_genes_retigene.csv (37.3 KB)
  data/raw/rp_genes_rivolta2025.csv (10.2 KB)


In [18]:
# ─────────────────────────────────────────────────────────────────
# Notebook 04 — Supervisor Confirmation Cell
#
# Supervisor confirmed on 2026-08-13:
#   "The lowest threshold is good since we will have to retain
#    interactions that are in the same compartment and tissue."
#
# Decision: Use score >= 0.400 as the working network.
# Biological filtering (compartment + tissue) will be applied
# in Stage 5 to reduce the network to retinal-relevant interactions.
# ─────────────────────────────────────────────────────────────────

print("=" * 55)
print("SUPERVISOR CONFIRMATION — 2026-08-13")
print("=" * 55)
print()
print("Confidence threshold : 0.400 (CONFIRMED)")
print("Rationale            : Broad network retained at low")
print("                       threshold. Biological filtering")
print("                       in Stage 5 will reduce network")
print("                       to retinal-relevant interactions")
print("                       using COMPARTMENTS and tissue")
print("                       expression databases.")
print()
print("Working network summary (final):")
print(f"  Score threshold : 0.400")
print(f"  Total edges     : 9,835")
print(f"  Total proteins  : 4,132")
print(f"    Seed proteins : 105")
print(f"    New partners  : 4,027")
print()
print("Next: Stage 5 — Context-specific filtering")
print("  Filter 1: COMPARTMENTS — subcellular co-localisation")
print("  Filter 2: Human Protein Atlas — retinal tissue expression")

SUPERVISOR CONFIRMATION — 2026-08-13

Confidence threshold : 0.400 (CONFIRMED)
Rationale            : Broad network retained at low
                       threshold. Biological filtering
                       in Stage 5 will reduce network
                       to retinal-relevant interactions
                       using COMPARTMENTS and tissue
                       expression databases.

Working network summary (final):
  Score threshold : 0.400
  Total edges     : 9,835
  Total proteins  : 4,132
    Seed proteins : 105
    New partners  : 4,027

Next: Stage 5 — Context-specific filtering
  Filter 1: COMPARTMENTS — subcellular co-localisation
  Filter 2: Human Protein Atlas — retinal tissue expression
